# Paperclip Literature Retrieval

![Paperclip Literature Retrieval](https://proto-bio.github.io/proto-assets/images/tool/paperclip/hero.png)

This notebook demonstrates the four Paperclip tools, which search and read biomedical literature through GXL's hosted Paperclip service. `run_paperclip_search` finds papers by topic, `run_paperclip_lookup` finds papers by an identifier such as a DOI or PMID, `run_paperclip_grep` finds every paper whose full text mentions an exact term, and `run_paperclip_document` reads part of one paper. Together they follow the workflow GXL recommends: find the relevant papers first, then read only the lines that matter and cite them by line number. Set `PAPERCLIP_API_KEY` in your environment before running.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("paperclip")
display_overview("paperclip")
display_docs_section("paperclip", "Background")

# Paperclip

[Paperclip](https://paperclip.gxl.ai/) is a hosted literature search and retrieval service from [GXL](https://gxl.ai/) covering more than 3.4 million papers and preprints. This toolkit exposes four tools over the Paperclip API that follow GXL's recommended workflow, namely `paperclip-literature-search` (find papers by topic), `paperclip-literature-lookup` (find papers by DOI, PMID, or another metadata field), `paperclip-literature-grep` (find papers whose full text mentions exact text), and `paperclip-literature-document` (read part of one paper). All four run on CPU in process and need only network access and a Paperclip API key.

[Paperclip](https://paperclip.gxl.ai/) is built by [GXL](https://gxl.ai/) to make the biomedical literature searchable and readable by researchers and AI agents. It indexes full-text papers from [PubMed Central](https://www.ncbi.nlm.nih.gov/pmc/) together with preprints from [bioRxiv](https://www.biorxiv.org/), [medRxiv](https://www.medrxiv.org/), and [arXiv](https://arxiv.org/), plus a separate abstract-level index spanning all of these sources. Searches combine keyword matching with matching by meaning, so a relevant passage can be found even when it is worded differently from the query. Every paper has a stable identifier, such as `PMC11754230`, and its full text is served with numbered lines, which makes it easy to cite the exact lines that support a claim.

GXL recommends finding papers first, then reading only the parts that matter. Search returns the most relevant papers for a topic, lookup resolves an identifier you already have, and grep finds every paper that mentions an exact term such as a gene name or accession. Reading a paper then means fetching a range of lines or the lines that match a pattern, rather than the whole text. All four tools query the live Paperclip service, so results reflect the literature at the time of the request. Each result also records where and when it was retrieved, so the same record can be traced back and checked later.

## Available tools

In [2]:
display_available_tools("paperclip")

- **`run_paperclip_document()`** — Read a Paperclip paper's metadata plus a line range, pattern matches, or full text
- **`run_paperclip_grep()`** — Find papers whose full text matches a regular expression across the Paperclip corpus
- **`run_paperclip_lookup()`** — Find Paperclip papers by DOI, PMID, PMC ID, arXiv ID, title, author, journal, or year
- **`run_paperclip_search()`** — Search Paperclip full-text and abstract literature corpora for citable papers

### `run_paperclip_search`

`run_paperclip_search` is the starting point for most literature questions. It takes a free-text query and ranks papers by a mix of keyword and meaning-based matching, so relevant work surfaces even when it uses different wording. Each hit carries its identifier, citation details, abstract, the passage that matched, and a relevance score. By default it searches PubMed Central, which GXL recommends for general biomedical literature.

In [3]:
from proto_tools import (
    PaperclipSearchConfig,
    PaperclipSearchInput,
    run_paperclip_search,
)

In [4]:
# Display docs
display_api_reference("paperclip", "input", "run_paperclip_search")

# Search for published work on designing binders for CAR T cells
inputs = PaperclipSearchInput(query="CAR T cell binder design")

**Input** — `PaperclipSearchInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>query</code> | <code>str</code> | required | Biomedical literature search query |

In [5]:
# Display config docs
display_api_reference("paperclip", "config", "run_paperclip_search")

# Return the top 5 hits from PubMed Central | see docs above for all fields
config = PaperclipSearchConfig(limit=5)

**Config** — `PaperclipSearchConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>paperclip_api_key</code> | <code>str &#124; None</code> | <code>$PAPERCLIP_API_KEY</code> | Paperclip API key. Falls back to $PAPERCLIP_API_KEY. |
| <code>paperclip_token</code> | <code>str &#124; None</code> | <code>$PAPERCLIP_TOKEN</code> | Paperclip bearer token, used when no API key is set. Falls back to $PAPERCLIP_TOKEN. |
| <code>timeout_seconds</code> | <code>float</code> | <code>120.0</code> | Maximum duration of one Paperclip service operation |
| <code>sources</code> | <code>list[Literal['pmc', 'papers', 'abstracts', 'biorxiv', 'medrxiv', 'arxiv']]</code> | <code>['pmc']</code> | Paperclip corpora to search; 'papers' covers all four full-text corpora |
| <code>limit</code> | <code>int</code> | <code>10</code> | Maximum search hits |
| <code>exact</code> | <code>bool</code> | <code>False</code> | Use Paperclip exact-text search |
| <code>since</code> | <code>str &#124; None</code> | <code>None</code> | Optional date lower bound for PMC, bioRxiv, and arXiv |
| <code>year</code> | <code>int &#124; None</code> | <code>None</code> | Only return papers from this year |
| <code>journal</code> | <code>str &#124; None</code> | <code>None</code> | Only return papers from this journal |
| <code>author</code> | <code>str &#124; None</code> | <code>None</code> | Only return papers by this author |
| <code>article_type</code> | <code>str &#124; None</code> | <code>None</code> | Only return papers of this article type |

In [6]:
# Run the search
search_output = run_paperclip_search(inputs, config)

In [7]:
# Display output docs
display_api_reference("paperclip", "output", "run_paperclip_search")

# Inspect the ranked hits
for hit in search_output.hits:
    print(f"{hit.document_id}  ({hit.year})  {hit.title[:80]}")
    if hit.matched_text:
        print(f"    matched: {hit.matched_text[:120]}...")

**Output** — `PaperclipSearchOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>query</code> | <code>str</code> | required | Submitted literature query |
| <code>sources</code> | <code>list[str]</code> | required | Searched Paperclip corpora |
| <code>result_id</code> | <code>str &#124; None</code> | <code>None</code> | Paperclip result identifier |
| <code>hits</code> | <code>list[PaperclipSearchHit]</code> | <code>[]</code> | Normalized literature matches |

**`PaperclipSearchHit`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>document_id</code> | <code>str</code> | required | Stable Paperclip document identifier |
| <code>title</code> | <code>str</code> | required | Publication title |
| <code>authors</code> | <code>str &#124; None</code> | <code>None</code> | Author list as Paperclip formats it |
| <code>journal</code> | <code>str &#124; None</code> | <code>None</code> | Journal or preprint server |
| <code>publication_date</code> | <code>str &#124; None</code> | <code>None</code> | Publication date |
| <code>year</code> | <code>int &#124; None</code> | <code>None</code> | Publication year |
| <code>doi</code> | <code>str &#124; None</code> | <code>None</code> | Digital object identifier |
| <code>pmid</code> | <code>str &#124; None</code> | <code>None</code> | PubMed identifier |
| <code>pmcid</code> | <code>str &#124; None</code> | <code>None</code> | PubMed Central identifier |
| <code>source</code> | <code>str</code> | required | Paperclip source corpus |
| <code>article_type</code> | <code>str &#124; None</code> | <code>None</code> | Article type, where recorded |
| <code>abstract</code> | <code>str &#124; None</code> | <code>None</code> | Publication abstract when available |
| <code>source_url</code> | <code>str</code> | required | Paperclip citation page for the paper |
| <code>matched_text</code> | <code>str &#124; None</code> | <code>None</code> | Text fragment matched by search |
| <code>relevance_score</code> | <code>float &#124; None</code> | <code>None</code> | Paperclip relevance score |
| <code>access_kind</code> | <code>LiteratureAccessKind</code> | required | Content granularity returned |

PMC10576856  (2023)  Bringing cell therapy to tumors: considerations for optimal CAR binder design
PMC13001334  (2026)  Validation and analysis of 12,000 AI-driven CAR-T designs in the Bits to Binders
PMC10983075  (2024)  Correction to: Bringing cell therapy to tumors: considerations for optimal CAR b
    matched: Correction to: Bringing cell therapy to tumors: considerations for optimal CAR binder design...
PMC12710756  (2025)  Sequence and structural determinants of efficacious de novo chimeric antigen rec
PMC13124637  (2026)  CAR T cell engineering approaches to minimise toxicities


#### Filtering by Corpus and Year

Metadata filters narrow a search more reliably than adding those words to the query. Here we search all four full-text corpora at once, including bioRxiv, medRxiv, and arXiv preprints, and keep only papers published in 2025.

In [8]:
# Search PMC and all preprint servers, restricted to 2025
recent_output = run_paperclip_search(
    PaperclipSearchInput(query="de novo protein binder design"),
    PaperclipSearchConfig(sources=["papers"], year=2025, limit=5),
)

for hit in recent_output.hits:
    print(f"{hit.document_id}  [{hit.source}]  {hit.title[:80]}")

bio_056f488783df  [biorxiv]  PXDesign: Fast, Modular, and Accurate De Novo Design of Protein Binders
bio_2914e0cf841a  [biorxiv]  Robust and Reliable  de novo  Protein Design: A Flow-Matching-Based Protein Gene
arx_2507.19375  [arxiv]  Latent-X: An Atom-level Frontier Model for De Novo Protein Binder Design
PMC12412469  [pmc]  A Complementarity‐Based Approach to De Novo Binder Design
bio_66acf00b763e  [biorxiv]  De novo protein design enables targeting of intractable oncogenic interfaces


### `run_paperclip_lookup`

When you already know which paper you want, `run_paperclip_lookup` finds it by a single metadata field: a DOI, PMID, PMC ID, arXiv ID, title, author, journal, or year. This bridges identifiers produced by other tools, such as a PMID from NCBI or a DOI cited in a UniProt entry, to the Paperclip identifier needed to read the paper.

In [9]:
from proto_tools import (
    PaperclipLookupConfig,
    PaperclipLookupInput,
    run_paperclip_lookup,
)

In [10]:
# Display docs
display_api_reference("paperclip", "input", "run_paperclip_lookup")

# Look up a paper by its PubMed Central ID
lookup_inputs = PaperclipLookupInput(field="pmc", value="PMC11754230")

**Input** — `PaperclipLookupInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>field</code> | <code>Literal['doi', 'pmid', 'pmc', 'arxiv', 'title', 'author', 'journal', 'year']</code> | required | Metadata field to match, such as doi or pmid |
| <code>value</code> | <code>str</code> | required | Value to look up in that field |

In [11]:
# Display config docs
display_api_reference("paperclip", "config", "run_paperclip_lookup")

# Use default config | see docs above for all fields
lookup_config = PaperclipLookupConfig()

**Config** — `PaperclipLookupConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>paperclip_api_key</code> | <code>str &#124; None</code> | <code>$PAPERCLIP_API_KEY</code> | Paperclip API key. Falls back to $PAPERCLIP_API_KEY. |
| <code>paperclip_token</code> | <code>str &#124; None</code> | <code>$PAPERCLIP_TOKEN</code> | Paperclip bearer token, used when no API key is set. Falls back to $PAPERCLIP_TOKEN. |
| <code>timeout_seconds</code> | <code>float</code> | <code>120.0</code> | Maximum duration of one Paperclip service operation |
| <code>limit</code> | <code>int</code> | <code>10</code> | Maximum papers returned |

In [12]:
# Run the lookup
lookup_output = run_paperclip_lookup(lookup_inputs, lookup_config)

In [13]:
# Display output docs
display_api_reference("paperclip", "output", "run_paperclip_lookup")

# Inspect the matching paper
for hit in lookup_output.hits:
    print(f"Document ID: {hit.document_id}")
    print(f"Title: {hit.title}")
    print(f"Journal: {hit.journal} ({hit.year})")
    print(f"DOI: {hit.doi}  PMID: {hit.pmid}")

**Output** — `PaperclipLookupOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>field</code> | <code>str</code> | required | Metadata field that was matched |
| <code>value</code> | <code>str</code> | required | Value that was looked up |
| <code>result_id</code> | <code>str &#124; None</code> | <code>None</code> | Paperclip result identifier |
| <code>hits</code> | <code>list[PaperclipSearchHit]</code> | <code>[]</code> | Matching papers |

**`PaperclipSearchHit`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>document_id</code> | <code>str</code> | required | Stable Paperclip document identifier |
| <code>title</code> | <code>str</code> | required | Publication title |
| <code>authors</code> | <code>str &#124; None</code> | <code>None</code> | Author list as Paperclip formats it |
| <code>journal</code> | <code>str &#124; None</code> | <code>None</code> | Journal or preprint server |
| <code>publication_date</code> | <code>str &#124; None</code> | <code>None</code> | Publication date |
| <code>year</code> | <code>int &#124; None</code> | <code>None</code> | Publication year |
| <code>doi</code> | <code>str &#124; None</code> | <code>None</code> | Digital object identifier |
| <code>pmid</code> | <code>str &#124; None</code> | <code>None</code> | PubMed identifier |
| <code>pmcid</code> | <code>str &#124; None</code> | <code>None</code> | PubMed Central identifier |
| <code>source</code> | <code>str</code> | required | Paperclip source corpus |
| <code>article_type</code> | <code>str &#124; None</code> | <code>None</code> | Article type, where recorded |
| <code>abstract</code> | <code>str &#124; None</code> | <code>None</code> | Publication abstract when available |
| <code>source_url</code> | <code>str</code> | required | Paperclip citation page for the paper |
| <code>matched_text</code> | <code>str &#124; None</code> | <code>None</code> | Text fragment matched by search |
| <code>relevance_score</code> | <code>float &#124; None</code> | <code>None</code> | Paperclip relevance score |
| <code>access_kind</code> | <code>LiteratureAccessKind</code> | required | Content granularity returned |

Document ID: PMC11754230
Title: CAR-T cell therapy: developments, challenges and expanded applications from cancer to autoimmunity
Journal: Frontiers in Immunology (2025)
DOI: 10.3389/fimmu.2024.1519671  PMID: 39850899


### `run_paperclip_grep`

`run_paperclip_grep` searches the full text of every paper for a regular expression. Unlike search, which ranks papers by relevance, grep finds every paper that mentions an exact term, including mentions buried in methods or data-availability sections. This makes it the right tool for tracing where a specific gene, tool, dataset, or accession has been used. Because the scan is time-bounded, repeated calls can return a different set or order of matches.

In [14]:
from proto_tools import (
    PaperclipGrepConfig,
    PaperclipGrepInput,
    run_paperclip_grep,
)

In [15]:
# Display docs
display_api_reference("paperclip", "input", "run_paperclip_grep")

# Find papers whose full text mentions ProteinMPNN
grep_inputs = PaperclipGrepInput(pattern="ProteinMPNN")

**Input** — `PaperclipGrepInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>pattern</code> | <code>str</code> | required | Regular expression matched against paper full text, such as a gene or accession |

In [16]:
# Display config docs
display_api_reference("paperclip", "config", "run_paperclip_grep")

# Return up to 20 matching paragraphs | see docs above for all fields
grep_config = PaperclipGrepConfig(max_matches=20)

**Config** — `PaperclipGrepConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>paperclip_api_key</code> | <code>str &#124; None</code> | <code>$PAPERCLIP_API_KEY</code> | Paperclip API key. Falls back to $PAPERCLIP_API_KEY. |
| <code>paperclip_token</code> | <code>str &#124; None</code> | <code>$PAPERCLIP_TOKEN</code> | Paperclip bearer token, used when no API key is set. Falls back to $PAPERCLIP_TOKEN. |
| <code>timeout_seconds</code> | <code>float</code> | <code>120.0</code> | Maximum duration of one Paperclip service operation |
| <code>max_matches</code> | <code>int</code> | <code>50</code> | Maximum matching paragraphs |
| <code>ignore_case</code> | <code>bool</code> | <code>False</code> | Match regardless of case |
| <code>exhaustive</code> | <code>bool</code> | <code>False</code> | Scan for the full time budget; use when a rare pattern returns nothing |

In [17]:
# Run the grep
grep_output = run_paperclip_grep(grep_inputs, grep_config)

In [18]:
# Display output docs
display_api_reference("paperclip", "output", "run_paperclip_grep")

# Inspect the papers and paragraphs that matched
papers = list(dict.fromkeys(match.document_id for match in grep_output.matches))
print(f"Papers with a match: {len(papers)}")
for match in grep_output.matches[:5]:
    print(f"{match.document_id} ({match.source}): {match.text[:100]}")

**Output** — `PaperclipGrepOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>pattern</code> | <code>str</code> | required | Submitted regular expression |
| <code>result_id</code> | <code>str &#124; None</code> | <code>None</code> | Paperclip result identifier |
| <code>matches</code> | <code>list[PaperclipGrepMatch]</code> | <code>[]</code> | Matching paragraphs, grouped by paper in match order |
| <code>source_url</code> | <code>str</code> | required | Paperclip service the corpus was searched on |

**`PaperclipGrepMatch`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>document_id</code> | <code>str</code> | required | Paper the paragraph is from |
| <code>source</code> | <code>str</code> | required | Corpus the paper belongs to |
| <code>text</code> | <code>str</code> | required | The matching paragraph |

Papers with a match: 20
arx_2605.11347 (arxiv): Most of the practically relevant objectives in protein design are non-differentiable or only availab
arx_2607.23518 (arxiv): • Baseline 1: RFDiffusion + ProteinMPNN. We generate separate binder backbones for each target using
arx_2609.05451 (arxiv): Net charge is a developability-relevant property of therapeutic binders, associated with viscosity, 
arx_2609.05436 (arxiv): …abCampath, P5A, and CAH2 scaffoldsMachine learning models with SVD, row-average features, and a pro
arx_2609.18745 (arxiv): …ras, Ivan Anishchenko, Nathaniel Bennett, Hua Bai, Robert J. Ragotte, Lukas F. Milles, Basile I. M.


### `run_paperclip_document`

`run_paperclip_document` reads one paper. It always returns the paper's metadata, and by default also reads its first 100 lines, which usually cover the title and abstract. Every line of a Paperclip paper is numbered, so evidence can be cited by line. GXL recommends reading a range of lines or the lines that match a pattern rather than the whole paper; full text is available with `read="full_text"` when you really need it.

In [19]:
from proto_tools import (
    PaperclipDocumentConfig,
    PaperclipDocumentInput,
    run_paperclip_document,
)

In [20]:
# Display docs
display_api_reference("paperclip", "input", "run_paperclip_document")

# Read the paper found by the lookup above
document_inputs = PaperclipDocumentInput(document_id="PMC11754230")

**Input** — `PaperclipDocumentInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>document_id</code> | <code>str</code> | required | Paperclip corpus identifier, such as PMC11754230 |

In [21]:
# Display config docs
display_api_reference("paperclip", "config", "run_paperclip_document")

# Read the first 40 lines | see docs above for all fields
document_config = PaperclipDocumentConfig(read="lines", start_line=1, end_line=40)

**Config** — `PaperclipDocumentConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>paperclip_api_key</code> | <code>str &#124; None</code> | <code>$PAPERCLIP_API_KEY</code> | Paperclip API key. Falls back to $PAPERCLIP_API_KEY. |
| <code>paperclip_token</code> | <code>str &#124; None</code> | <code>$PAPERCLIP_TOKEN</code> | Paperclip bearer token, used when no API key is set. Falls back to $PAPERCLIP_TOKEN. |
| <code>timeout_seconds</code> | <code>float</code> | <code>120.0</code> | Maximum duration of one Paperclip service operation |
| <code>read</code> | <code>Literal['metadata', 'lines', 'grep', 'full_text']</code> | <code>'lines'</code> | What to read beyond metadata; GXL recommends lines or grep over full_text |
| <code>start_line</code> | <code>int</code> | <code>1</code> | First line read in lines mode |
| <code>end_line</code> | <code>int</code> | <code>100</code> | Last line read in lines mode |
| <code>pattern</code> | <code>str &#124; None</code> | <code>None</code> | Regular expression to find in grep mode |
| <code>context_lines</code> | <code>int</code> | <code>0</code> | Lines of context around each grep match |
| <code>ignore_case</code> | <code>bool</code> | <code>False</code> | Match grep patterns regardless of case |

In [22]:
# Run the read
document_output = run_paperclip_document(document_inputs, document_config)

In [23]:
# Display output docs
display_api_reference("paperclip", "output", "run_paperclip_document")

# Inspect the metadata and the excerpt
document = document_output.document
print(f"Title: {document.paper.title}")
print(f"Access kind: {document.access_kind}")
print(document.excerpt[:800] if document.excerpt else "No text returned")

**Output** — `PaperclipDocumentOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>document</code> | <code>PaperclipDocument</code> | required | Retrieved literature document |

**`PaperclipDocument`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>paper</code> | <code>PaperclipPaper</code> | required | Citation details for the paper |
| <code>access_kind</code> | <code>LiteratureAccessKind</code> | required | Content granularity returned |
| <code>excerpt</code> | <code>str &#124; None</code> | <code>None</code> | Requested line range as Paperclip returned it |
| <code>matches</code> | <code>list[PaperclipLineMatch]</code> | <code>[]</code> | Lines matched in grep mode, with context lines |
| <code>content</code> | <code>str &#124; None</code> | <code>None</code> | Complete line-numbered full text |
| <code>content_preview</code> | <code>str &#124; None</code> | <code>None</code> | Incomplete full text returned by Paperclip |
| <code>line_count</code> | <code>int &#124; None</code> | <code>None</code> | Lines Paperclip reports for the full text |

**`PaperclipPaper`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>document_id</code> | <code>str</code> | required | Stable Paperclip document identifier |
| <code>title</code> | <code>str</code> | required | Publication title |
| <code>authors</code> | <code>str &#124; None</code> | <code>None</code> | Author list as Paperclip formats it |
| <code>journal</code> | <code>str &#124; None</code> | <code>None</code> | Journal or preprint server |
| <code>publication_date</code> | <code>str &#124; None</code> | <code>None</code> | Publication date |
| <code>year</code> | <code>int &#124; None</code> | <code>None</code> | Publication year |
| <code>doi</code> | <code>str &#124; None</code> | <code>None</code> | Digital object identifier |
| <code>pmid</code> | <code>str &#124; None</code> | <code>None</code> | PubMed identifier |
| <code>pmcid</code> | <code>str &#124; None</code> | <code>None</code> | PubMed Central identifier |
| <code>source</code> | <code>str</code> | required | Paperclip source corpus |
| <code>article_type</code> | <code>str &#124; None</code> | <code>None</code> | Article type, where recorded |
| <code>abstract</code> | <code>str &#124; None</code> | <code>None</code> | Publication abstract when available |
| <code>source_url</code> | <code>str</code> | required | Paperclip citation page for the paper |

**`PaperclipLineMatch`**

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>line_number</code> | <code>int</code> | required | Line number in the paper, for citations |
| <code>text</code> | <code>str</code> | required | Text of the line |
| <code>is_match</code> | <code>bool</code> | required | True for a matching line, False for a context line |

Title: CAR-T cell therapy: developments, challenges and expanded applications from cancer to autoimmunity
Access kind: excerpt
L1: CAR-T cell therapy: developments, challenges and expanded applications from cancer to autoimmunity
L2: Frontiers in Immunology (2025)
L3: Yaojie Kong, Jingyao Li, Xueyao Zhao, Yanwei Wu, Liang Chen
L4: School of Medicine, Shanghai University, Shanghai, China
L5: Chimeric Antigen Receptor (CAR)-T cell therapy has rapidly emerged as a groundbreaking approach in cancer treatment, particularly for hematologic malignancies. However, the application of CAR-T cell therapy in solid tumors remains challenging. This review summarized the development of CAR-T technologies, emphasized the challenges and solutions in CAR-T cell therapy for solid tumors. Also, key innovations were discussed including specialized CAR-T, combination therapies and the novel use of CAR-Treg, CAR-NK and CAR-M cells. Besi


#### Reading the Lines That Match a Pattern

Setting `read="grep"` returns only the lines that match a pattern, each with its line number, plus optional surrounding context lines. This is the most efficient way to pull a specific measurement or claim out of a long paper, and the line numbers go straight into a citation.

In [24]:
# Find every mention of binding affinity, with one line of context on each side
affinity_output = run_paperclip_document(
    document_inputs,
    PaperclipDocumentConfig(read="grep", pattern="affinity|K_?[dD]", ignore_case=True, context_lines=1),
)

for match in affinity_output.document.matches:
    marker = ">" if match.is_match else " "
    print(f"{marker} L{match.line_number}: {match.text[:120]}")

  L19: The antigen recognition domain consists of an ScFv formed by linking the variable heavy chain (VH) and light chain (VL) 
> L20: Importantly, the recognition of target cells by CAR-T cell rely strongly on the high affinity of the ScFv component for 
> L21: However, it is important to note that the affinity of the ScFv for the target antigen and its efficacy for CAR-T are not
> L22: Almost all CAR constructs use ScFvs derived from monoclonal antibodies. However, the low folding stability of VH and VL 
  L23: Hinge region
  L67: Earlier generations of CAR-T cell therapies, while effective, face challenges such as immune rejection by the host and t
> L68: Compared with autologous CAR-T cells, allogeneic CAR-T cells possess distinct advantages, including broad individual com
  L69: The design, preparation, and application of CAR-T cells exhibit considerable heterogeneity, as their effectiveness varie
  L96: A notable obstacle to the widespread adoption of CAR-T cell therapy is the to

#### From Search to Evidence

The tools are designed to chain. Here we take the top hit from the first search and pull the lines from it that mention CAR T cells, producing line-numbered evidence that can be cited directly.

In [25]:
top_hit = search_output.hits[0]
evidence = run_paperclip_document(
    PaperclipDocumentInput(document_id=top_hit.document_id),
    PaperclipDocumentConfig(read="grep", pattern="CAR[- ]?T", ignore_case=True),
).document

print(f"{top_hit.title}\n")
for match in evidence.matches[:10]:
    print(f"L{match.line_number}: {match.text[:120]}")

Bringing cell therapy to tumors: considerations for optimal CAR binder design

L9: Chimeric antigen receptor (CAR)-T cells have shown impressive success in the clinic, with six approved therapies across 
L10: Table 1: Summary of CAR-T therapies approved by the US Food and Drug Administration
L27: Figure 1: The modular architecture of chimeric antigen receptors. The various generations of CARs are defined by their i
L30: In broad strokes, the considerations for identifying a suitable target for cell therapy in oncology are like those for a
L33: While many CARs have utilized scFvs as their binding domain, several alternate scaffolds have also been used that potent
L34: The marketed CAR-T ciltacabtagene autoleucel (Carvykti) utilizes two nanobody domains that recognize distinct non-overla
L35: Binding domains from native proteins have also been successfully used in CARs, where the tumor-specific or tumor-enriche
L36: As the NK-cell receptor NKG2D recognizes a range of ligands that are exp

## Export Results

Search results and documents can be saved to JSON, including each paper's citation URL, for downstream use in other tools or analysis pipelines. Each paper keeps its own copyright and license, so this example writes to a temporary directory rather than committing retrieved text.

In [26]:
from pathlib import Path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as output_dir:
    search_output.export("paperclip_search", export_path=output_dir, file_format="json")
    document_output.export("paperclip_document", export_path=output_dir, file_format="json")
    print(f"Exported {sorted(path.name for path in Path(output_dir).iterdir())}")

Exported ['paperclip_document.json', 'paperclip_search.json']
